# 12 · Classify with Enhancements
**Full classification with all pipeline enhancements enabled on a 100-row sample.**

| Enhancement | What it adds |
|-------------|-------------|
| Confidence  | `_confidence_score` per label; routes low-confidence rows to a review queue |
| Feedback    | Injects human-validated examples into the prompt |
| Hierarchical | Forces L2 codes to match their declared L1 parent |
| Incremental | Skips rows already classified (checkpoint-based) |
| Cost tracker | Logs token usage and $ cost per LLM call |
| Language    | Detects non-English text; translates before classification |
| Quality     | Generates a per-run quality report comparing to previous run |

### Policy taxonomy harmonisation — CAR Manual v3.37 + Content Coverage Guide (June 2026)

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|------|-------|--------|
| PA10 | Article-in-Press (AiP) eligibility and processing | CAR Manual §5 |
| PA11 | Excluded item types (full CAR §1 exclusion list) | CAR Manual §1 |
| PU2.1–PU2.5 | Customer root cause group | Merged from `policy_understanding` into `policy_compliance` |
| CR1.8 | AiP visibility or missing (contact reason) | CAR Manual §5 |

**Modules changed:**
- `policy_compliance`: expanded with PA10/PA11 policy areas + two new fields (`customer_understanding_issue`, `customer_root_cause_code`)
- `policy_understanding`: **disabled** — its customer dimension fields are now produced by `policy_compliance` in a single LLM pass

**`MODULES` list updated to include `policy_compliance`.** Output will contain `customer_understanding_issue` and `customer_root_cause_code` columns alongside existing `agent_response_compliance` and `applicable_policy_codes`. Re-run `rca build --force-all` (notebook 11) before classifying.

### PA12–PA17 additions (June 2026) — first classification run required

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|---|---|---|
| PA12 | Patent indexing and coverage | Content Coverage Guide §2.3 |
| PA13 | Author identity and profile management | Content Coverage Guide §3.3 |
| PA14 | Affiliation profile curation | Content Coverage Guide §3.4 |
| PA15 | Open Access classification | Content Coverage Guide §3.1 |
| PA16 | Errata and record corrections | CCG §3.1 + CAR Manual §6 |
| PA17 | Cited references coverage and linkage | Content Coverage Guide §3.2, §4.5 |

After updating the Excel, rebuild the module:
`rca build build --force policy_compliance`

**Incremental strategy (Option A) — keeps existing PA1–PA11 counts stable:**

To avoid changing PA1–PA11 ticket counts, re-classify only tickets currently assigned PA0 ("No applicable policy"). Merge the re-classified PA0 subset back into the existing enriched Excel.

```python
# Load existing enriched Excel, filter to PA0 tickets only
df_existing = pd.read_excel("runs/output/Scopus_Content_Policy_enriched.xlsx")
df_pa0 = df_existing[df_existing["applicable_policy_codes_1"] == "PA0"].copy()
# Run classification on df_pa0 → write to a separate file
# Then merge: update only PA0 rows in df_existing with the new classifications
```

The R analysis script (`scopus_effort_master_v1.R`) will automatically handle the new codes — `policy_doc_map` already contains entries for PA12–PA17, and `fct_lump_min(min = 50)` in the model handles sparse new codes gracefully.


In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import asyncio, json
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

# ── Settings ─────────────────────────────────────────────────────────────
DATA_FILE    = "data/Scopus_Content_Collapsed.xlsx"
OUTPUT_FILE  = "runs/output/enhanced_sample_enriched.xlsx"
TAXONOMY_DIR = "generated/taxonomies"

ID_COL       = "Reference #"
TEXT_COL     = "Conversation"
SAMPLE_N     = 100
SAMPLE_SEED  = 42

MODEL        = "azure/dsaic-llm-general"
BATCH_SIZE   = 10
CONCURRENCY  = 2
RETRIES      = 3
MAX_TOKENS   = 4096
MODULES      = ["contact_reason", "customer_effort", "policy_compliance"]

# Enhancement toggles
ENABLE_CONFIDENCE    = True
CONFIDENCE_THRESHOLD = 0.70
EXPORT_REVIEW_QUEUE  = True

ENABLE_FEEDBACK      = True
FEEDBACK_DIR         = "runs/feedback"
FEEDBACK_MAX         = 5
FEEDBACK_STRATEGY    = "smart"    # recent | diverse | frequent | smart

ENABLE_HIERARCHICAL  = True

ENABLE_INCREMENTAL   = True        # set True to use checkpoint
CHECKPOINT_FILE      = "runs/checkpoints/enhanced_sample_checkpoint.jsonl"
BAD_UIDS_FILE        = "runs/checkpoints/enhanced_sample_bad_uids.txt"

TRACK_COSTS          = True
COST_REPORT_DIR      = "runs/cost_reports"

DETECT_LANGUAGE      = True        # True = detect + translate non-English
LANGUAGE_STRATEGY    = "translate"

RUN_QUALITY_CHECK    = True
QUALITY_REPORT_DIR   = "runs/output"

## Step 1 · Load and sample data

In [ ]:
df_full = pd.read_excel(DATA_FILE)
df = df_full.sample(n=min(SAMPLE_N, len(df_full)), random_state=SAMPLE_SEED).reset_index(drop=True)
df[ID_COL] = df[ID_COL].astype(str)

print(f"Full dataset : {len(df_full):,} rows")
print(f"Sample       : {len(df):,} rows")
df[[ID_COL, TEXT_COL]].head(2)

## Step 2 · Configure RunConfig with all enhancements

In [ ]:
from rca_pipeline.config import RunConfig
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import drop_enrichment_cols
from rca_pipeline.pipeline import prepare_batched_records

# Write sample to temp file
Path("runs/output").mkdir(parents=True, exist_ok=True)
Path("runs/checkpoints").mkdir(parents=True, exist_ok=True)
_tmp = "runs/output/_enhanced_tmp.xlsx"
df.to_excel(_tmp, index=False)

cfg = RunConfig(
    input_file   = _tmp,
    output_file  = OUTPUT_FILE,
    taxonomy_dir = TAXONOMY_DIR,
    id_col       = ID_COL,
    text_col     = TEXT_COL,
    modules      = MODULES,
    model        = MODEL,
    batch_size   = BATCH_SIZE,
    concurrency  = CONCURRENCY,
    retries      = RETRIES,
    max_tokens   = MAX_TOKENS,

    enable_confidence    = ENABLE_CONFIDENCE,
    confidence_threshold = CONFIDENCE_THRESHOLD,
    export_review_queue  = EXPORT_REVIEW_QUEUE,

    feedback_dir          = FEEDBACK_DIR,
    feedback_max_examples = FEEDBACK_MAX,
    feedback_strategy     = FEEDBACK_STRATEGY,

    hierarchical_modules = MODULES if ENABLE_HIERARCHICAL else [],

    incremental            = ENABLE_INCREMENTAL,
    incremental_state_file = CHECKPOINT_FILE,

    track_costs     = TRACK_COSTS,
    cost_report_dir = COST_REPORT_DIR,

    detect_language  = DETECT_LANGUAGE,
    language_strategy = LANGUAGE_STRATEGY,

    run_quality_check    = RUN_QUALITY_CHECK,
    quality_report_dir   = QUALITY_REPORT_DIR,
    previous_quality_report = None,
)

taxonomies = load_taxonomies(TAXONOMY_DIR)
df_work    = drop_enrichment_cols(df.copy())
batches    = prepare_batched_records(df_work, cfg)

print(f"Batches: {len(batches)}")
print(f"Enhancements active:")
print(f"  confidence={ENABLE_CONFIDENCE}, feedback={ENABLE_FEEDBACK},"
      f" hierarchical={ENABLE_HIERARCHICAL}")
print(f"  incremental={ENABLE_INCREMENTAL}, cost_track={TRACK_COSTS},"
      f" language={DETECT_LANGUAGE}, quality={RUN_QUALITY_CHECK}")

## Step 3 · Run classification

In [ ]:
import time
from rca_pipeline.pipeline import run_classification_module, MODULE_REGISTRY
from rca_pipeline.enhancements.confidence import (
    enrich_with_confidence, inject_confidence_instruction,
)
from rca_pipeline.enhancements.cost_tracker import CostTracker
from rca_pipeline.llm_client import (
    set_system_prompt_suffix, clear_system_prompt_suffix,
    register_cost_tracker, unregister_cost_tracker, set_cost_tracker_module,
)

all_results = []

# Register cost tracker before any LLM calls
cost_tracker = None
if TRACK_COSTS:
    cost_tracker = CostTracker(model=MODEL)
    register_cost_tracker(cost_tracker)

async def _classify():
    for mod in MODULES:
        if mod not in MODULE_REGISTRY:
            print(f"  skipping {mod} — not in registry")
            continue

        # Tell the cost tracker which module is running
        if TRACK_COSTS:
            set_cost_tracker_module(mod)

        # Build the system prompt suffix for this module
        suffix_parts = []
        if ENABLE_CONFIDENCE:
            suffix_parts.append(inject_confidence_instruction(""))
        set_system_prompt_suffix("\n".join(suffix_parts))

        t0 = time.time()
        results = await run_classification_module(mod, batches, taxonomies, cfg)
        clear_system_prompt_suffix()

        if ENABLE_CONFIDENCE:
            results = enrich_with_confidence(results, threshold=CONFIDENCE_THRESHOLD)
            n_flagged = sum(1 for r in results if r.get("_needs_review", False))
            print(f"  {mod:<22}  {len(results)} records  {time.time()-t0:.1f}s  "
                  f"({n_flagged} below confidence threshold)")
        else:
            print(f"  {mod:<22}  {len(results)} records  {time.time()-t0:.1f}s")

        all_results.extend(results)

print("Classifying...")
asyncio.run(_classify())

# Save cost report and unregister tracker
if cost_tracker is not None:
    unregister_cost_tracker()
    Path(COST_REPORT_DIR).mkdir(parents=True, exist_ok=True)
    report_path = Path(COST_REPORT_DIR) / f"cost_{cost_tracker._run_id}.json"
    cost_tracker.save(report_path)
    cost_tracker.print_summary()

print(f"\nDone — {len(all_results)} result records total")

## Step 4 · Post-process and inspect enhancements

In [ ]:
from rca_pipeline.utils import post_process, write_output_file

grouped = {}
for r in all_results:
    uid = str(r.get("uid", ""))
    grouped.setdefault(uid, {"uid": uid}).update(r)

df_out = post_process(df_work, list(grouped.values()), taxonomies, cfg)
write_output_file(df_out, OUTPUT_FILE)

print(f"Output: {len(df_out)} rows × {len(df_out.columns)} columns → {OUTPUT_FILE}")
enriched_cols = [c for c in df_out.columns if c not in df_work.columns]
print(f"New columns ({len(enriched_cols)}): {enriched_cols}")

In [ ]:
# Confidence scores
# enrich_with_confidence adds '_confidence' (float) and '_needs_review' (bool) per record
conf_cols = [c for c in df_out.columns if "confidence" in c.lower()]
review_col = "_needs_review" if "_needs_review" in df_out.columns else None

if conf_cols:
    print("Confidence score distribution:")
    print(df_out[conf_cols].describe().round(3).to_string())
    print()
    primary_conf = "_confidence" if "_confidence" in df_out.columns else conf_cols[0]
    valid_scores = df_out[primary_conf].dropna()
    below = (valid_scores < CONFIDENCE_THRESHOLD).sum()
    missing = df_out[primary_conf].isna().sum()
    print(f"Scores available  : {len(valid_scores)}/{len(df_out)}")
    print(f"Below threshold ({CONFIDENCE_THRESHOLD}): {below} rows → review queue")
    if missing:
        print(f"Missing scores    : {missing} rows (LLM did not self-report confidence)")
elif not ENABLE_CONFIDENCE:
    print("Confidence disabled (ENABLE_CONFIDENCE=False).")
else:
    print("No confidence columns found — the LLM may not have returned a 'confidence' field.")
    print("This can happen if the model ignores the self-report instruction.")
    print("Check the raw results with: [r.get('confidence') for r in all_results[:5]]")
    print([r.get("confidence") for r in all_results[:5]])

In [ ]:
# Review queue (low-confidence rows)
review_path = Path(OUTPUT_FILE).parent / "review_queue.xlsx"
if review_path.exists():
    rq = pd.read_excel(review_path)
    print(f"Review queue: {len(rq)} rows in {review_path}")
    display(rq.head(5))
elif conf_cols:
    # Build it manually if not written separately
    rq = df_out[df_out[conf_cols[0]] < CONFIDENCE_THRESHOLD][[ID_COL, TEXT_COL] + conf_cols[:2]]
    print(f"Low-confidence rows ({len(rq)}):")
    display(rq.head(5))

In [ ]:
# Cost report
cost_files = sorted(Path(COST_REPORT_DIR).glob("cost_*.json")) if Path(COST_REPORT_DIR).exists() else []
if cost_files:
    latest = json.loads(cost_files[-1].read_text(encoding="utf-8"))
    summary = latest.get("summary", {})
    by_mod  = latest.get("by_module", {})
    print(f"Latest cost report: {cost_files[-1].name}")
    print(f"  model          : {summary.get('model', '?')}")
    print(f"  total records  : {summary.get('total_records', '?'):,}")
    print(f"  total tokens   : {summary.get('total_tokens', 0):,}  "
          f"(prompt {summary.get('total_prompt_tokens',0):,} + "
          f"completion {summary.get('total_completion_tokens',0):,})")
    print(f"  total cost     : ${summary.get('total_cost_usd', 0):.4f}")
    print(f"  avg/record     : ${summary.get('avg_cost_per_record', 0):.6f}")
    print(f"  elapsed        : {summary.get('elapsed_s', 0):.1f}s  "
          f"({summary.get('records_per_second', 0):.1f} rec/s)")
    print()
    print(f"  {'Module':<22}  {'Records':>8}  {'Tokens':>10}  {'Cost':>10}  {'$/rec':>10}")
    print(f"  {'-'*22}  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}")
    for mod, m in sorted(by_mod.items()):
        print(f"  {mod:<22}  {m['total_records']:>8,}  "
              f"{m['total_tokens']:>10,}  "
              f"${m['total_cost_usd']:>9.4f}  "
              f"${m['avg_cost_per_record']:>9.6f}")

    # Cost projection to full dataset
    if summary.get("total_records", 0) > 0:
        full_n = 8307  # Scopus full dataset size
        proj = summary["total_cost_usd"] / summary["total_records"] * full_n
        print(f"\n  Full dataset projection ({full_n:,} rows): ${proj:.2f}")
else:
    print("No cost reports found (cost tracking may not have written yet).")

In [ ]:
# Quality report
quality_files = sorted(Path(QUALITY_REPORT_DIR).glob("quality_*.json")) if Path(QUALITY_REPORT_DIR).exists() else []
if not quality_files:
    print("No quality reports found.")
else:
    qr = json.loads(quality_files[-1].read_text(encoding="utf-8"))

    print("=" * 60)
    print("  QUALITY REPORT")
    print("=" * 60)
    print(f"  Generated : {qr.get('generated_at', '?')}")
    print(f"  Records   : {qr.get('total_records', '?'):,}")
    print()

    # ── Coverage ──────────────────────────────────────────────
    cov = qr.get("coverage", {})
    summary = cov.pop("_summary", {})
    print(f"  COVERAGE  (avg {summary.get('avg_coverage_pct','?')}%)")
    print(f"  {'-'*56}")
    for field, s in cov.items():
        status_icon = "✓" if s.get("status") == "good" else "✗"
        print(f"  {status_icon} {field:<40} {s.get('coverage_pct','?'):>5}%")
    print()

    # ── Code distribution ─────────────────────────────────────
    dist = qr.get("code_distribution", {})
    for field, d in dist.items():
        alerts = d.get("alerts", [])
        print(f"  CODE DISTRIBUTION  {field}  ({d.get('unique_codes','?')} codes)")
        print(f"  {'-'*56}")
        for code, info in list(d.get("codes", {}).items())[:10]:
            bar = "█" * int(info["pct"] / 2)
            print(f"  {code:<8} {info['label'][:28]:<28} {info['pct']:>5.1f}%  {bar}")
        if len(d.get("codes", {})) > 10:
            print(f"           ... {len(d['codes'])-10} more codes")
        if alerts:
            print()
            for alert in alerts:
                print(f"  ⚠  {alert}")
        print()

    # ── Evidence quality ──────────────────────────────────────
    ev = qr.get("evidence_quality", {})
    if ev:
        print(f"  EVIDENCE GROUNDING")
        print(f"  {'-'*56}")
        for field, s in ev.items():
            icon = "✓" if s.get("status") == "good" else "⚠"
            print(f"  {icon} {field:<40} {s.get('grounding_rate_pct','?'):>5.1f}%"
                  f"  ({s.get('ungrounded',0)} ungrounded)")
        print()

    # ── Error rate ────────────────────────────────────────────
    err = qr.get("error_rate", {})
    if err:
        icon = "✓" if err.get("status") == "good" else "✗"
        print(f"  ERROR RATE")
        print(f"  {'-'*56}")
        print(f"  {icon} {err.get('error_rate_pct', 0):.1f}% errors"
              f"  ({err.get('error_records',0)} / {err.get('total_records',0)} records)")
        print()

    # ── Confidence ────────────────────────────────────────────
    conf = qr.get("confidence", {})
    if conf.get("enabled"):
        print(f"  CONFIDENCE SCORES")
        print(f"  {'-'*56}")
        print(f"  mean {conf.get('mean','?'):.3f}  "
              f"median {conf.get('median','?'):.3f}  "
              f"std {conf.get('std','?'):.3f}  "
              f"min {conf.get('min','?'):.2f}")
        print()

    # ── Language ──────────────────────────────────────────────
    lang = qr.get("language", {})
    if lang.get("enabled"):
        print(f"  LANGUAGE DISTRIBUTION  ({lang.get('non_english_pct','?')}% non-English)")
        print(f"  {'-'*56}")
        for code, info in lang.get("distribution", {}).items():
            bar = "█" * int(info["pct"] / 2)
            print(f"  {code:<6} {info['count']:>5,} records  {info['pct']:>5.1f}%  {bar}")
        print()

    # ── Anomalies ─────────────────────────────────────────────
    anomalies = qr.get("anomalies", [])
    if anomalies:
        print(f"  ANOMALIES  ({len(anomalies)} detected)")
        print(f"  {'-'*56}")
        for a in anomalies:
            print(f"  ⚠  {a}")
        print()
    else:
        print(f"  ANOMALIES")
        print(f"  {'-'*56}")
        print(f"  ✓ None detected")
        print()

    print("=" * 60)